# Set up

This notebooks is based in the preprocessing notebook made for the research work [Machine Learning, Clinical Notes and Knowledge Graphs for Early Prediction of Acute Kidney Injury in the Intensive Care](https://bitbucket.org/aumc-kik/ml-cn-kg-4-aki-prediction/src/master/)

In [1]:
from datetime import datetime

print(datetime.now())
# data preprocessing
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import collections
from collections import defaultdict
import os
import sys
import shutil
from collections import Counter

2025-03-01 12:15:25.932637


The following files were extracted by running the queries specified in the following repositories:
* https://github.com/mapo89/continuous-aki-predict/tree/main
* https://github.com/MIT-LCP/mimic-code

Some modifications were made to update those queries to mimic version 4

In [2]:
DATA_PATH_stages = "../data/aki_preprocessing/kdigo_stages_measured.csv"
DATA_PATH_labs = "../data/aki_preprocessing/labs-kdigo_stages_measured.csv"
DATA_PATH_vitals = "../data/aki_preprocessing/vitals-kdigo_stages_measured.csv"
DATA_PATH_vents = "../data/aki_preprocessing/vents-vasopressor-sedatives-kdigo_stages_measured.csv"
DATA_PATH_detail = "../data/aki_preprocessing/icustay_detail-kdigo_stages_measured.csv"
SEPARATOR = ";"

In [3]:
# the output pathes are here
OUTPUT_PATH = "data/AKI_fts2"

In [4]:
# Set parameter as constant

# which classifier to use, only run one classifier at one time
ALL_STAGES = False  # not binary label, each class separately 0,1,2,3

CLASS1 = True  # AnyAKI
CLASS2 = False  # ModerateSevereAKI
CLASS3 = False  # SevereAKI


MAX_FEATURE_SET = True

# resampling  and imputing
TIME_SAMPLING = True
SAMPLING_INTERVAL = "1H"
# RESAMPLE_LIMIT = 16 # 4 days*6h interval

# if MOST_COMMON is not applied,sampling with different strategies per kind of variable,
# numeric variables use mean value, categorical variables use max value
MOST_COMMON = False  # resampling with most common

# fit Yereva's time span
MAX_HOUR = 48

# How much time the prediction should occur (hours)
HOURS_AHEAD = 48

IMPUTE_EACH_ID = True  # imputation within each icustay_id with most common value | False like in the original notebook
IMPUTE_COLUMN = False  # imputation based on whole column
IMPUTE_METHOD = "most_frequent"
FILL_VALUE = 0  # fill missing value and ragged part of 3d array

# Age constraints: adults
ADULTS_MIN_AGE = 18
ADULTS_MAX_AGE = -1

NORMALIZATION = "min-max"
NORM_TYPE = "min_max"

CAPPING = True
if CAPPING:
    CAPPING_THRESHOLD_UPPER = 0.99
    CAPPING_THRESHOLD_LOWER = 0.01


# use random split or fixed train/val/test set
RANDOM_SPLIT = True
FIXED = False
RANDOM_SEED = 42
SPLIT_SIZE = 0.2

# set changable info corresponding to each classifier as variables

min_set = ["icustay_id", "charttime", "creat", "uo_rt_6hr", "uo_rt_12hr", "uo_rt_24hr", "aki_stage"]

max_set = [
    "icustay_id",
    "charttime",
    "aki_stage",
    "hadm_id",
    #"albumin_avg", # Not in original notebook
    "aniongap_avg",
    "bicarbonate_avg",
    "bilirubin_avg",
    #"bun_avg", # Not in original notebook
    "chloride_avg",
    "creat",
    "diasbp_mean",
    "glucose_avg",
    "heartrate_mean",
    "hematocrit_avg",
    "hemoglobin_avg",
    "potassium_avg",
    "resprate_mean",
    "sodium_avg",
    "spo2_mean",
    "sysbp_mean",
    "uo_rt_12hr",
    "uo_rt_24hr",
    "uo_rt_6hr",
    "wbc_avg",
    "sedative",
    "vasopressor",
    "vent",
    "age",
    "F",
    "M",
    "asian",
    "black",
    "hispanic",
    "native",
    "other",
    "unknown",
    "white",
    "ELECTIVE",
    "EMERGENCY",
    "URGENT",
]
print(f'Max set: {len(max_set)}')

Max set: 39


In [5]:
# Some functions used later
def cap_data(df):
    print("Capping between the {} and {} quantile".format(CAPPING_THRESHOLD_LOWER, CAPPING_THRESHOLD_UPPER))

    cap_mask = df.columns.difference(['icustay_id', 'charttime', 'aki_stage'])

    # Filtrar solo columnas numéricas
    numeric_cols = df[cap_mask].select_dtypes(include=[np.number]).columns

    df[numeric_cols] = df[numeric_cols].clip(
        df[numeric_cols].quantile(CAPPING_THRESHOLD_LOWER),
        df[numeric_cols].quantile(CAPPING_THRESHOLD_UPPER),
        axis=1
    )

    return df

def normalise_data(df, norm_mask):
    df[norm_mask] = (df[norm_mask] - df[norm_mask].min()) / (df[norm_mask].max() - df[norm_mask].min())
    return df


# impute missing value in resampleing data with most common based on each id
def fast_mode(df, key_cols, value_col):
    """Calculate a column mode, by group, ignoring null values.

    key_cols : list of str - Columns to groupby for calculation of mode.
    value_col : str - Column for which to calculate the mode.

    Return
    pandas.DataFrame
        One row for the mode of value_col per key_cols group. If ties, returns the one which is sorted first."""
    return (
        df.groupby(key_cols + [value_col])
        .size()
        .to_frame("counts")
        .reset_index()
        .sort_values("counts", ascending=False)
        .drop_duplicates(subset=key_cols)
    ).drop("counts", axis=1)


# get max shape of 3d array
def get_dimensions(array, level=0):
    yield level, len(array)
    try:
        for row in array:
            yield from get_dimensions(row, level + 1)
    except TypeError:  # not an iterable
        pass


def get_max_shape(array):
    dimensions = defaultdict(int)
    for level, length in get_dimensions(array):
        dimensions[level] = max(dimensions[level], length)
    return [value for _, value in sorted(dimensions.items())]


# pad the ragged 3d array to rectangular shape based on max size
def iterate_nested_array(array, index=()):
    try:
        for idx, row in enumerate(array):
            yield from iterate_nested_array(row, (*index, idx))
    except TypeError:  # final level
        yield (*index, slice(len(array))), array  # think of the types


def pad(array, fill_value):
    dimensions = get_max_shape(array)
    result = np.full(dimensions, fill_value, dtype=np.float64)
    for index, value in iterate_nested_array(array):
        result[index] = value
    return result

# Read csv files

## kdigo_stages_measured

In [6]:
print("Reading "+DATA_PATH_stages+"...")

# Reading csv files
X = pd.read_csv(DATA_PATH_stages, sep=SEPARATOR)
X.drop(["aki_stage_creat", "aki_stage_uo"], axis=1, inplace=True)

print("Remove rows where all the values are missing")
initial_row_count = X.shape[0]
X = X.dropna(how="all", subset=["creat", "uo_rt_6hr", "uo_rt_12hr", "uo_rt_24hr", "aki_stage"])
removed_row_count = initial_row_count - X.shape[0]
print(f"Number of rows removed: {removed_row_count}")

print("Convert charttime to timestamp")
X["charttime"] = pd.to_datetime(X["charttime"])

# merge rows if they have exact timestamp within same icustay_id AL : it substitutes missing values with zero
# X = X.groupby(['icustay_id', 'charttime']).sum().reset_index(['icustay_id', 'charttime'])

unique_subjects = X['subject_id'].nunique()
print(f"Unique subject_id count: {unique_subjects}")
print(f"Unique stay_id count: {X['stay_id'].nunique()}")

X.head()

Reading ../data/aki_preprocessing/kdigo_stages_measured.csv...
Remove rows where all the values are missing
Number of rows removed: 0
Convert charttime to timestamp
Unique subject_id count: 50878
Unique stay_id count: 73092


,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,creat,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,aki_stage,aki_stage_smoothed
0,10000032,29079034,39553978,2180-07-23 06:39:00,NaN,NaN,0.7,NaN,NaN,NaN,NaN,0,0
1,10000032,29079034,39553978,2180-07-23 15:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,10000032,29079034,39553978,2180-07-23 21:45:00,0.7,0.7,0.5,NaN,NaN,NaN,NaN,0,0
3,10000980,26913865,39765666,2189-06-27 06:48:00,NaN,NaN,2.3,NaN,NaN,NaN,NaN,0,0
4,10000980,26913865,39765666,2189-06-27 09:08:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [7]:
patients_per_aki_stage = X.groupby('aki_stage')['subject_id'].nunique()
print(patients_per_aki_stage)

aki_stage
0    50878
1    33282
2    24507
3     8891
Name: subject_id, dtype: int64


## icustay_detail-kdigo_stages_measured

In [8]:
print("Reading "+DATA_PATH_detail+"...")
dataset_detail = pd.read_csv(DATA_PATH_detail, sep=SEPARATOR)  # age constraint
# keep "intime" to calculate Hours in Yereva

# subject_id;hadm_id;stay_id;gender;anchor_age;anchor_year;anchor_year_group;admittime;dischtime;deathtime;
# race -> ethnicity
# deathtime -> dod

dataset_detail.drop(
    [
        "dod",
        "admittime",
        "dischtime",
        "los_hospital",
        'ethnicity', # In original notebook
        #"race", # Not in original notebook
        "hospital_expire_flag",
        "hospstay_seq",
        "first_hosp_stay",
        #"intime", # In original notebook
        "outtime",
        "los_icu",
        "icustay_seq",
        "first_icu_stay",
    ],
    axis=1,
    inplace=True,
    errors="ignore",
)

dataset_detail.head()

Reading ../data/aki_preprocessing/icustay_detail-kdigo_stages_measured.csv...


,subject_id,hadm_id,stay_id,gender,admission_age,race,icu_intime,icu_outtime,subject_id.1,gender.1,...,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race.1,edregtime,edouttime,hospital_expire_flag.1
0,10000032,29079034,39553978,F,52.559969,WHITE,2180-07-23 14:00:00,2180-07-23 23:50:47,10000032,F,...,P30KEH,EMERGENCY ROOM,HOME,Medicaid,ENGLISH,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
1,10000980,26913865,39765666,F,76.486231,BLACK/AFRICAN AMERICAN,2189-06-27 08:42:00,2189-06-27 20:38:27,10000980,F,...,P30KEH,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,ENGLISH,MARRIED,BLACK/AFRICAN AMERICAN,2189-06-27 06:25:00,2189-06-27 08:42:00,0
2,10001217,24597018,37067082,F,55.881486,WHITE,2157-11-20 19:18:02,2157-11-21 22:08:00,10001217,F,...,P4645A,EMERGENCY ROOM,HOME HEALTH CARE,Other,?,MARRIED,WHITE,2157-11-18 17:38:00,2157-11-19 01:24:00,0
3,10001217,27703517,34592300,F,55.962942,WHITE,2157-12-19 15:42:24,2157-12-20 14:27:41,10001217,F,...,P99698,PHYSICIAN REFERRAL,HOME HEALTH CARE,Other,?,MARRIED,WHITE,NaN,NaN,0
4,10001725,25563031,31205490,F,46.275517,WHITE,2110-04-11 15:52:22,2110-04-12 23:59:56,10001725,F,...,P35SU0,PACU,HOME,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,0


In [9]:
print(f"Unique subject_id count: {dataset_detail['subject_id'].nunique()}")

Unique subject_id count: 50878


In [10]:
dataset_detail.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'gender', 'admission_age', 'race',
       'icu_intime', 'icu_outtime', 'subject_id.1', 'gender.1', 'anchor_age',
       'anchor_year', 'anchor_year_group', 'dod.1', 'subject_id.2',
       'hadm_id.1', 'admittime.1', 'dischtime.1', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status',
       'race.1', 'edregtime', 'edouttime', 'hospital_expire_flag.1'],
      dtype='object')

Variables duplicated
* subject_id, subject_id.1, subject_id.2
* hadm_id, hadm_id.1
* gender, gender.1
* dod (Previously removed), dod.1
* admittime(Previously removed), admittime.1
* dischtime(Prev removed), dischtime.1
* race, race.1
* hospital_expire_flag(Prev removed), hospital_expire_flag.1

As there are rows duplicated, we check if there's any difference among them by running the following cell:

In [11]:
inconsistent_rows = dataset_detail[['race', 'race.1']].apply(lambda row: row.nunique() != 1, axis=1)
if inconsistent_rows.any():
    print("There are inconsistent rows in the dataset.")
else:
    print("All rows are consistent.")

All rows are consistent.


After checking the columns are the same, remove the duplicates

In [12]:
# Remove duplicated columns, and the ones that are copy of previous removed columns
dataset_detail.drop(
    ['subject_id.1','subject_id.2','hadm_id.1','gender.1','dod.1','admittime.1','dischtime.1','race.1','hospital_expire_flag.1'],
    axis=1,
    inplace=True,
    errors="ignore",
)
dataset_detail.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'gender', 'admission_age', 'race',
       'icu_intime', 'icu_outtime', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'deathtime', 'admission_type', 'admit_provider_id',
       'admission_location', 'discharge_location', 'insurance', 'language',
       'marital_status', 'edregtime', 'edouttime'],
      dtype='object')

In [13]:
print("convert intime to timestamp")
dataset_detail["intime"] = pd.to_datetime(dataset_detail["icu_intime"])

INTIME = pd.DataFrame()
INTIME["stay_id"] = dataset_detail["stay_id"]
INTIME["intime"] = dataset_detail["intime"]

convert intime to timestamp


## labs-kdigo_stages_measured

In [14]:
print("Reading "+DATA_PATH_labs+"...")
dataset_labs = pd.read_csv(DATA_PATH_labs, sep=SEPARATOR)  # 'bands lactate platelet ptt inr pt
dataset_labs.drop(
    [
        "albumin_min",
        "albumin_max",
        "bilirubin_min",
        "bilirubin_max",
        "bands_min",
        "bands_max",
        "lactate_min",
        "lactate_max",
        "platelet_min",
        "platelet_max",
        "ptt_min",
        "ptt_max",
        "inr_min",
        "inr_max",
        "pt_min",
        "pt_max",
    ],
    axis=1,
    inplace=True,
)
dataset_labs.head()

Reading ../data/aki_preprocessing/labs-kdigo_stages_measured.csv...


,subject_id,hadm_id,stay_id,charttime,aniongap_min,aniongap_max,bicarbonate_min,bicarbonate_max,creatinine_min,creatinine_max,...,hemoglobin_min,hemoglobin_max,potassium_min,potassium_max,sodium_min,sodium_max,bun_min,bun_max,wbc_min,wbc_max
0,12466550,23998182,30000153,2174-09-29 12:27:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12466550,23998182,30000153,2174-09-29 13:27:00,NaN,NaN,NaN,NaN,NaN,NaN,...,12.5,12.5,4.4,4.4,141.0,141.0,NaN,NaN,NaN,NaN
2,12466550,23998182,30000153,2174-09-29 14:07:00,NaN,NaN,NaN,NaN,NaN,NaN,...,10.9,10.9,4.2,4.2,142.0,142.0,NaN,NaN,NaN,NaN
3,12466550,23998182,30000153,2174-09-29 15:37:00,12.0,12.0,19.0,19.0,0.9,0.9,...,10.8,10.8,4.4,4.4,142.0,142.0,22.0,22.0,17.0,17.0
4,12466550,23998182,30000153,2174-09-29 16:05:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
initial_size = dataset_labs.shape[0]
print("Initial number of rows: ", initial_size)
print("Drop rows with missing charttime")
dataset_labs = dataset_labs.dropna(subset=["charttime"])
print(f"Number of rows removed: {initial_size - dataset_labs.shape[0]}")

initial_size = dataset_labs.shape[0]
print("Drop rows with missing values in all columns")
dataset_labs = dataset_labs.dropna(subset=dataset_labs.columns[4:], how="all")
print(f"Number of rows removed: {initial_size - dataset_labs.shape[0]}")

print("Convert charttime to timestamp")
dataset_labs["charttime"] = pd.to_datetime(dataset_labs["charttime"])
dataset_labs = dataset_labs.sort_values(by=["stay_id", "charttime"])

print("Final number of rows: ", dataset_labs.shape[0])

Initial number of rows:  1908141
Drop rows with missing charttime
Number of rows removed: 614
Drop rows with missing values in all columns
Number of rows removed: 357391
Convert charttime to timestamp
Final number of rows:  1550136


In [16]:
print("dataset_labs shape: ", dataset_labs.shape)
print("Unique subject_id: ",dataset_labs['subject_id'].nunique())
print("Unique stay_id: ",dataset_labs['stay_id'].nunique())
dataset_labs.columns

dataset_labs shape:  (1550136, 26)
Unique subject_id:  50399
Unique stay_id:  72424


Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'aniongap_min',
       'aniongap_max', 'bicarbonate_min', 'bicarbonate_max', 'creatinine_min',
       'creatinine_max', 'chloride_min', 'chloride_max', 'glucose_min',
       'glucose_max', 'hematocrit_min', 'hematocrit_max', 'hemoglobin_min',
       'hemoglobin_max', 'potassium_min', 'potassium_max', 'sodium_min',
       'sodium_max', 'bun_min', 'bun_max', 'wbc_min', 'wbc_max'],
      dtype='object')

## vitals-kdigo_stages_measured & vents-vasopressor-sedatives-kdigo_stages_measured

In [17]:
print("Reading "+DATA_PATH_vitals+" and "+DATA_PATH_vents+"...")
if MAX_FEATURE_SET:
    dataset_vitals = pd.read_csv(DATA_PATH_vitals, sep=SEPARATOR)
    dataset_vents = pd.read_csv(DATA_PATH_vents, sep=SEPARATOR)
    # dataset_icd = pd.read_csv(DATA_PATH_icd, sep= SEPARATOR)

    print("Drop columns that will not be used")
    dataset_vitals.drop(
        [
            "heartrate_min",
            "heartrate_max",
            "sysbp_min",
            "sysbp_max",
            "diasbp_min",
            "diasbp_max",
            "meanbp_min",
            "meanbp_max",
            "meanbp_mean",
            "tempc_min",
            "tempc_max",
            "tempc_mean",
            "resprate_min",
            "resprate_max",
            "spo2_min",
            "spo2_max",
            "glucose_min",
            "glucose_max",
        ],
        axis=1,
        inplace=True,
    )
    print("convert charttime to timestamp in both dataframes")
    dataset_vitals["charttime"] = pd.to_datetime(dataset_vitals["charttime"])
    dataset_vents["charttime"] = pd.to_datetime(dataset_vents["charttime"])

    print("sort values by stay_id and charttime")
    dataset_vitals = dataset_vitals.sort_values(by=["stay_id", "charttime"])
    dataset_vents = dataset_vents.sort_values(by=["stay_id", "charttime"])
    # AL drop those where all columns are nan (empty rows)
    initial_size = dataset_vitals.shape[0]
    print("\nInitial number of rows in dataset_vitals: ", initial_size)
    print("Drop rows with missing charttime")
    dataset_vitals = dataset_vitals.dropna(subset=dataset_vitals.columns[4:], how="all")
    print(f"Number of rows removed: {initial_size - dataset_vitals.shape[0]}")

print("\ndataset_vitals shape: ", dataset_vitals.shape)
print("Unique subject_id: ",dataset_vitals['subject_id'].nunique())
print("Unique stay_id: ",dataset_vitals['stay_id'].nunique())
dataset_vitals.head()

Reading ../data/aki_preprocessing/vitals-kdigo_stages_measured.csv and ../data/aki_preprocessing/vents-vasopressor-sedatives-kdigo_stages_measured.csv...
Drop columns that will not be used
convert charttime to timestamp in both dataframes
sort values by stay_id and charttime

Initial number of rows in dataset_vitals:  9735110
Drop rows with missing charttime
Number of rows removed: 108632

dataset_vitals shape:  (9626478, 10)
Unique subject_id:  50878
Unique stay_id:  73083


,subject_id,hadm_id,stay_id,charttime,heartrate_mean,sysbp_mean,diasbp_mean,resprate_mean,spo2_mean,glucose_mean
2391773,12466550,23998182,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,14.0,NaN,NaN
2391774,12466550,23998182,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,100.0,NaN
2391775,12466550,23998182,30000153,2174-09-29 12:06:00,100.0,136.0,74.0,NaN,NaN,NaN
2391777,12466550,23998182,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,18.0,NaN,NaN
2391778,12466550,23998182,30000153,2174-09-29 13:00:00,104.0,113.0,77.0,16.0,100.0,NaN


In [18]:
print("dataset_vents shape: ", dataset_vents.shape)
print("Unique stay_id: ",dataset_vents['stay_id'].nunique())
dataset_vents.head()

dataset_vents shape:  (5524789, 5)
Unique stay_id:  73092


,stay_id,charttime,vent,vasopressor,sedative
0,30000153,2174-09-29 10:16:00,0,0,0
1,30000153,2174-09-29 12:12:00,1,0,1
2,30000153,2174-09-29 12:27:00,1,0,1
3,30000153,2174-09-29 13:27:00,1,0,1
4,30000153,2174-09-29 14:00:00,1,0,0


# Calculate avg of each Lab feature

In [19]:
print("compute avg from min/max in labs file")
print(datetime.now())
# Labs file: instead of min and max their avg
counter = 0
col1 = 4
col2 = 5
null_l = []  # no null values in those that are different
changed = 0  # 4316 records changed to avg

'''
11 pairs of columns to be averaged:
    'aniongap_min','aniongap_max', 
    'bicarbonate_min', 'bicarbonate_max',
    'creatinine_min','creatinine_max',
    'chloride_min', 'chloride_max',
    'glucose_min', 'glucose_max',
    'hematocrit_min', 'hematocrit_max',
    'hemoglobin_min', 'hemoglobin_max',
    'potassium_min', 'potassium_max',
    'sodium_min', 'sodium_max',
    'bun_min', 'bun_max',
    'wbc_min', 'wbc_max'
'''

while counter < 11:
    row = 0
    # find where min and max are different and save their row indices
    # Calculate avg between min and max. SAVE IT IN THE min column, remove the max column,
    # and continue with the next pair of columns.
    while row < len(dataset_labs):
        a = dataset_labs.iloc[row, col1]
        b = dataset_labs.iloc[row, col2]
        if a == b or (np.isnan(a) and np.isnan(b)):
            pass
        elif a != b:
            changed += 1
            avg = (a + b) / 2
            dataset_labs.iloc[row, col1] = avg
            if (np.isnan(a) and ~np.isnan(b)) or (np.isnan(b) and ~np.isnan(a)):
                null_l.append(row)
        else:
            print(a)
            print(b)
        row += 1
    # delete the redundant column max, update counters
    dataset_labs.drop(dataset_labs.columns[col2], axis=1, inplace=True)
    counter = counter + 1
    col1 = col1 + 1
    col2 = col2 + 1

dataset_labs.columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "charttime",
    "aniongap_avg",
    "bicarbonate_avg",
    "creatinine_avg",
    "chloride_avg",
    "glucose_avg",
    "hematocrit_avg",
    "hemoglobin_avg",
    "potassium_avg",
    "sodium_avg",
    "bun_avg",  # Blood Urea Nitrogen
    "wbc_avg",  # White blood cells
]
if len(null_l) > 0:
    print("null values encountered ", len(null_l))
else:
    print("no null values encountered")
print(datetime.now())

compute avg from min/max in labs file
2025-03-01 12:16:27.135410
no null values encountered
2025-03-01 12:28:08.246520


In [20]:
dataset_labs.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'aniongap_avg',
       'bicarbonate_avg', 'creatinine_avg', 'chloride_avg', 'glucose_avg',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'sodium_avg',
       'bun_avg', 'wbc_avg'],
      dtype='object')

# Merge creatinine 

In [21]:
print("Merge creatinine")
# merge creatinine from labs and set with labels
creatinine_lab = dataset_labs[["stay_id", "charttime", "creatinine_avg"]].copy()
creatinine_lab = creatinine_lab.dropna(subset=["creatinine_avg"])

creat = X[["stay_id", "charttime", "creat"]].copy()
creat = creat.dropna(subset=["creat"])

creatinine_lab = creatinine_lab.rename(columns={"creatinine_avg": "creat"})
# creat = creat.append(creatinine_lab, ignore_index=True) # for old version of pandas

print("creat shape from labs : ", creatinine_lab.shape)
print("creat shape from X: ", creat.shape)

creat = pd.concat([creat, creatinine_lab], ignore_index=True)
creat.drop_duplicates(inplace=True)

print("Final creat shape: ", creat.shape)
creat.head()

Merge creatinine
creat shape from labs :  (1040973, 3)
creat shape from X:  (599409, 3)
Final creat shape:  (1102090, 3)


,stay_id,charttime,creat
0,39553978,2180-07-23 06:39:00,0.7
1,39553978,2180-07-23 21:45:00,0.5
2,39765666,2189-06-27 06:48:00,2.3
3,37067082,2157-11-18 18:30:00,0.6
4,37067082,2157-11-20 08:14:00,0.7


In [22]:
print("dataset_labs shape from X: ", dataset_labs.shape)
print("Remove creatinine from labs")
dataset_labs.drop(["creatinine_avg"], axis=1, inplace=True)

print("Remove rows with missing values in all columns")
initial_size = dataset_labs.shape[0]
dataset_labs = dataset_labs.dropna(subset=dataset_labs.columns[4:], how="all")
print(f"Number of rows removed: {initial_size - dataset_labs.shape[0]}")
print("dataset_labs shape from X: ", dataset_labs.shape)
dataset_labs.head()

dataset_labs shape from X:  (1550136, 15)
Remove creatinine from labs
Remove rows with missing values in all columns
Number of rows removed: 718
dataset_labs shape from X:  (1549418, 14)


,subject_id,hadm_id,stay_id,charttime,aniongap_avg,bicarbonate_avg,chloride_avg,glucose_avg,hematocrit_avg,hemoglobin_avg,potassium_avg,sodium_avg,bun_avg,wbc_avg
0,12466550,23998182,30000153,2174-09-29 12:27:00,NaN,NaN,NaN,NaN,35.0,NaN,NaN,NaN,NaN,NaN
1,12466550,23998182,30000153,2174-09-29 13:27:00,NaN,NaN,110.0,158.0,38.0,12.5,4.4,141.0,NaN,NaN
2,12466550,23998182,30000153,2174-09-29 14:07:00,NaN,NaN,112.0,176.0,33.0,10.9,4.2,142.0,NaN,NaN
3,12466550,23998182,30000153,2174-09-29 15:37:00,12.0,19.0,115.0,192.0,31.7,10.8,4.4,142.0,22.0,17.0
4,12466550,23998182,30000153,2174-09-29 16:05:00,NaN,NaN,NaN,175.0,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
print("Remove creatinine from X")
X.drop(["creat"], axis=1, inplace=True)

print("Merge creatinine resulted form the previous operation in X")
X = pd.merge(X, creat, on=["stay_id", "charttime"], sort=True, how="outer", copy=False)
X.head()

Remove creatinine from X
Merge creatinine resulted form the previous operation in X


,subject_id,hadm_id,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,aki_stage,aki_stage_smoothed,creat
0,12466550.0,23998182.0,30000153,2174-09-29 10:16:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.2
1,12466550.0,23998182.0,30000153,2174-09-29 12:12:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
2,12466550.0,23998182.0,30000153,2174-09-29 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
3,12466550.0,23998182.0,30000153,2174-09-29 15:00:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
4,12466550.0,23998182.0,30000153,2174-09-29 15:37:00,1.2,1.2,NaN,NaN,NaN,NaN,0.0,0.0,0.9


# Merge glucose

Merge glucose from vitals and labs

In [24]:
dataset_vitals.head()

,subject_id,hadm_id,stay_id,charttime,heartrate_mean,sysbp_mean,diasbp_mean,resprate_mean,spo2_mean,glucose_mean
2391773,12466550,23998182,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,14.0,NaN,NaN
2391774,12466550,23998182,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,100.0,NaN
2391775,12466550,23998182,30000153,2174-09-29 12:06:00,100.0,136.0,74.0,NaN,NaN,NaN
2391777,12466550,23998182,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,18.0,NaN,NaN
2391778,12466550,23998182,30000153,2174-09-29 13:00:00,104.0,113.0,77.0,16.0,100.0,NaN


In [25]:
if MAX_FEATURE_SET:
    glucose_v = dataset_vitals[["subject_id", "hadm_id", "stay_id", "charttime", "glucose_mean"]].copy()
    glucose_v = glucose_v.dropna(subset=["glucose_mean"])
    glucose = dataset_labs[["subject_id", "hadm_id", "stay_id", "charttime", "glucose_avg"]].copy()
    glucose = glucose.dropna(subset=["glucose_avg"])
    glucose_v = glucose_v.rename(columns={"glucose_mean": "glucose_avg"})

    # glucose = glucose.append(glucose_v, ignore_index=True) # for old version of pandas
    glucose = pd.concat([glucose, glucose_v], ignore_index=True)

    glucose.drop_duplicates(inplace=True)
    
    # delete old columns
    dataset_labs.drop(["glucose_avg"], axis=1, inplace=True)
    dataset_vitals.drop(["glucose_mean"], axis=1, inplace=True)
    dataset_vitals = dataset_vitals.dropna(subset=dataset_vitals.columns[4:], how="all")
    
    # merge new column
    dataset_labs = pd.merge(
        dataset_labs,
        glucose,
        on=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "charttime",
        ],
        sort=True,
        how="outer",
        copy=False,
    )

dataset_labs = dataset_labs.sort_values(by=["stay_id", "charttime"], ignore_index=True)
X = X.sort_values(by=["stay_id", "charttime"], ignore_index=True)

print("Glucose from labs and vitals merged")
dataset_labs.head()

Glucose from labs and vitals merged


,subject_id,hadm_id,stay_id,charttime,aniongap_avg,bicarbonate_avg,chloride_avg,hematocrit_avg,hemoglobin_avg,potassium_avg,sodium_avg,bun_avg,wbc_avg,glucose_avg
0,12466550,23998182,30000153,2174-09-29 12:27:00,NaN,NaN,NaN,35.0,NaN,NaN,NaN,NaN,NaN,NaN
1,12466550,23998182,30000153,2174-09-29 13:27:00,NaN,NaN,110.0,38.0,12.5,4.4,141.0,NaN,NaN,158.0
2,12466550,23998182,30000153,2174-09-29 14:07:00,NaN,NaN,112.0,33.0,10.9,4.2,142.0,NaN,NaN,176.0
3,12466550,23998182,30000153,2174-09-29 15:37:00,12.0,19.0,115.0,31.7,10.8,4.4,142.0,22.0,17.0,192.0
4,12466550,23998182,30000153,2174-09-29 16:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,175.0


In [26]:
# Integrate labs data in the main dataframe
X = pd.merge(X, dataset_labs, on=["stay_id", "charttime"], how="outer", copy=False)

In [27]:
X.columns

Index(['subject_id_x', 'hadm_id_x', 'stay_id', 'charttime',
       'creat_low_past_7day', 'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr',
       'uo_rt_24hr', 'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed',
       'creat', 'subject_id_y', 'hadm_id_y', 'aniongap_avg', 'bicarbonate_avg',
       'chloride_avg', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg',
       'sodium_avg', 'bun_avg', 'wbc_avg', 'glucose_avg'],
      dtype='object')

In [28]:
'''
As in this step, X contains repeated columns, we need to check if there are any differences between them.
If there are, we need to decide how to handle them. Otherwise, we can remove the duplicate columns.
'''

# Initialize an array to store the indices of rows where subject_id_x and subject_id_y differ
differing_indices = []

# Iterate over the rows of the DataFrame
for index, row in X.iterrows():
    if pd.isna(row['subject_id_x']) and not pd.isna(row['subject_id_y']):
        X.at[index, 'subject_id_x'] = row['subject_id_y']
    elif not pd.isna(row['subject_id_x']) and not pd.isna(row['subject_id_y']):
        if row['subject_id_x'] != row['subject_id_y']:
            differing_indices.append(index)

# Print the indices of rows where subject_id_x and subject_id_y differ
print("Indices with differing subject_id values:", differing_indices)
# Initialize an array to store the indices of rows where hadm_id_x and hadm_id_y differ
differing_hadm_indices = []

# Iterate over the rows of the DataFrame
for index, row in X.iterrows():
    if pd.isna(row['hadm_id_x']) and not pd.isna(row['hadm_id_y']):
        X.at[index, 'hadm_id_x'] = row['hadm_id_y']
    elif not pd.isna(row['hadm_id_x']) and not pd.isna(row['hadm_id_y']):
        if row['hadm_id_x'] != row['hadm_id_y']:
            differing_hadm_indices.append(index)

# Print the indices of rows where hadm_id_x and hadm_id_y differ
print("Indices with differing hadm_id values:", differing_hadm_indices)

Indices with differing subject_id values: []
Indices with differing hadm_id values: []


In [29]:
# Drop the columns subject_id_y and hadm_id_y
X.drop(columns=['subject_id_y', 'hadm_id_y'], inplace=True)

# Rename the columns hadm_id_x and subject_id_x by removing the '_x'
X.rename(columns={'hadm_id_x': 'hadm_id', 'subject_id_x': 'subject_id'}, inplace=True)

# Display the updated columns
X.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'creat_low_past_7day',
       'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr',
       'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed', 'creat',
       'aniongap_avg', 'bicarbonate_avg', 'chloride_avg', 'hematocrit_avg',
       'hemoglobin_avg', 'potassium_avg', 'sodium_avg', 'bun_avg', 'wbc_avg',
       'glucose_avg'],
      dtype='object')

# Merge vital signs

In [30]:
X = pd.merge(X, dataset_vitals, on=["stay_id", "charttime", "subject_id", "hadm_id"], how="outer", copy=False)

In [31]:
X.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'charttime', 'creat_low_past_7day',
       'creat_low_past_48hr', 'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr',
       'aki_stage_crrt', 'aki_stage', 'aki_stage_smoothed', 'creat',
       'aniongap_avg', 'bicarbonate_avg', 'chloride_avg', 'hematocrit_avg',
       'hemoglobin_avg', 'potassium_avg', 'sodium_avg', 'bun_avg', 'wbc_avg',
       'glucose_avg', 'heartrate_mean', 'sysbp_mean', 'diasbp_mean',
       'resprate_mean', 'spo2_mean'],
      dtype='object')

In [32]:
X = pd.merge(X, dataset_vents, on=["stay_id", "charttime"], how="outer", copy=False)

# Removing patients under the min age

In [33]:
print("start preprocessing time dependent data")

print("Filter out patients with age less than 18 in dataset_detail")
initial_size = dataset_detail.shape[0]
print(f"Initial number of rows: {initial_size}")
dataset_detail = dataset_detail.loc[dataset_detail["anchor_age"] >= ADULTS_MIN_AGE] #admission_age
print(f"Number of rows removed: {initial_size - dataset_detail.shape[0]}")

adults_icustay_id_list = dataset_detail["stay_id"].unique()
X = X[X.stay_id.isin(adults_icustay_id_list)].sort_values(by=["stay_id"], ignore_index=True)
X = X.sort_values(by=["stay_id", "charttime"], ignore_index=True)
adults_icustay_id_list = np.sort(adults_icustay_id_list)

start preprocessing time dependent data
Filter out patients with age less than 18 in dataset_detail
Initial number of rows: 73092
Number of rows removed: 0


In [34]:
print(f"Number of unique icustay_id: {len(adults_icustay_id_list)}")

Number of unique icustay_id: 73092


In [35]:
# Temporal data preprocessing
# X_copy_2 = X.copy()
# X.columns

#Retrieve data
# X = X_copy_2.copy()

# Remove stay_id with less than 48 hrs

In [36]:
print("drop stay_id with time span less than 48hrs")

def more_than_HOURS_ahead(adults_icustay_id_list, X):
    drop_list = []
    los_list = []  # calculating LOS (Length of Stay) ICU based on charttime
    long_stays_id = []  # LOS longer than MAX DAYS days
    last_charttime_list = []

    # Sian modified to above code, AL: seq_length = X.groupby(['stay_id'],as_index=False).size().to_frame('size')
    seq_length = X.groupby(["stay_id"], as_index=False).size()
    
    id_count = 0
    first_row_index = 0

    while id_count < len(adults_icustay_id_list):
        stay_id = adults_icustay_id_list[id_count]
        last_row_index = (
            first_row_index + seq_length.iloc[id_count, 1] - 1
        )  # Sian modified, AL: seq_length.iloc[id_count,0]-1
        first_time = X.iat[first_row_index, X.columns.get_loc("charttime")]
        last_time = X.iat[last_row_index, X.columns.get_loc("charttime")]
        los = round(float((last_time - first_time).total_seconds() / 60 / 60 / 24), 4)  # in days
        if los < 48 / 24:
            drop_list.append(stay_id)
        else:
            los_list.append(los)
            if los > 35:
                long_stays_id.append(stay_id)
                last_charttime_list.append(last_time)
        # udpate for the next stay_id
        first_row_index = last_row_index + 1
        id_count += 1

    if len(long_stays_id) != len(last_charttime_list):
        print("ERROR")
        
    print("%d long stays (>35 days)" % len(long_stays_id))
    # drop all the rows with the saved stay_id
    print("there are %d id-s shorter than 48 hours" % len(drop_list))
    X = X[~X.stay_id.isin(drop_list)]
    id_list = X["stay_id"].unique()
    X = X.sort_values(by=["stay_id", "charttime"], ignore_index=True)

    return id_list, X, long_stays_id, last_charttime_list

# id_list: list of unique icustay_id
id_list, X, long_stays_id, last_charttime_list = more_than_HOURS_ahead(adults_icustay_id_list, X)

long = pd.DataFrame()
long["stay_id"] = long_stays_id
long["last_time"] = last_charttime_list

drop stay_id with time span less than 48hrs
3183 long stays (>35 days)
there are 7679 id-s shorter than 48 hours


In [41]:
print(f"Unique stay_if {len(id_list)}")

unique_patients = X['subject_id'].nunique()
print(f"Unique patients in X: {unique_patients}")

Unique stay_if 65413
Unique patients in X: 45233


# Extract Label 

In [42]:
print("Binarise labels")
if ALL_STAGES:
    pass
elif CLASS1:
    # No AKI = 0, AKI = 1
    X.loc[X['aki_stage'] > 1, 'aki_stage'] = 1
elif CLASS2:
    X.loc[X['aki_stage'] < 2, 'aki_stage'] = 0
    X.loc[X['aki_stage'] > 1, 'aki_stage'] = 1
elif CLASS3:
    X.loc[X['aki_stage'] < 3, 'aki_stage'] = 0
    X.loc[X['aki_stage'] > 2, 'aki_stage'] = 1

Binarise labels


In [43]:
# If any of the records presents aki_stage equal to 1, the target value for the stay_id will be 1.
# Otherwise, it will be 0.
print("Choose one label for each stay_id")

def one_label_per_icustay(id_list, X):
    dataset = X
    temp_icustay_df = pd.DataFrame()
    target_list = []

    for icustay in id_list:
        temp_icustay_df = dataset.loc[dataset["stay_id"] == icustay].sort_values(by=["charttime"])
        if any(temp_icustay_df.aki_stage == 1):
            target_list.append(1)
        else:
            target_list.append(0)

    return target_list


target_list = one_label_per_icustay(id_list, X)

target = pd.DataFrame()
target["stay_id"] = id_list
target["y_true"] = target_list

target.head()

Choose one label for each stay_id (whenever it turn pos in the whole staying)


,stay_id,y_true
0,30000153,1
1,30000213,1
2,30000484,1
3,30000646,0
4,30001148,1


In [44]:
print("Unique stay_id:", len(id_list))
y_true_count = target["y_true"].value_counts()
print("# of stay_id with AKI:", y_true_count[1])
print("# of stay_id without AKI:", y_true_count[0])

Unique stay_id: 65413
# of stay_id with AKI: 44654
# of stay_id without AKI: 20759


In [45]:
print("calculate how many positive label within the first 48hrs, could be different time span")

def count_positive_label(X, INTIME, hour):
    dataset = X
    dataset = pd.merge(dataset, INTIME, on=["stay_id"], how="left", copy=False)
    dataset["HOURS"] = (dataset.charttime - dataset.intime).apply(lambda s: s / np.timedelta64(1, "s")) / 60.0 / 60
    dataset = dataset[dataset["HOURS"] >= 0]
    dataset = dataset[dataset["HOURS"] <= hour]
    dataset = dataset.reset_index(drop=True)

    temp_icustay_df = pd.DataFrame()
    target_list = []

    for icustay in id_list:
        temp_icustay_df = dataset.loc[dataset["stay_id"] == icustay].sort_values(by=["charttime"])
        if any(temp_icustay_df.aki_stage == 1):
            target_list.append(1)
        else:
            target_list.append(0)
    print("number of neg and pos label within the first " + str(hour) + "hr")
    print(Counter(target_list))


# TODO 3/24: also compute within 24 hours
hour = 24
count_positive_label(X, INTIME, hour)

hour = 48  # set time span
count_positive_label(X, INTIME, hour)

calculate how many positive label within the first 48hrs, could be different time span
number of neg and pos label within the first 24hr
Counter({0: 33035, 1: 32378})
number of neg and pos label within the first 48hr
Counter({1: 39875, 0: 25538})


# ⭐ Checkpoint #1

In [76]:
import pickle

In [ ]:
# Save the DataFrame to a pickle file
with open('data/X.pkl', 'wb') as file:
    pickle.dump(X, file)

# Save the INTIME DataFrame to a pickle file
with open('data/INTIME.pkl', 'wb') as file:
    pickle.dump(INTIME, file)

# save dataset_detail
with open('data/dataset_detail.pkl', 'wb') as file:
    pickle.dump(dataset_detail, file)

# save target
with open('data/target.pkl', 'wb') as file:
    pickle.dump(target, file)

In [225]:
# Load the DataFrame from the pickle file
with open('data/X.pkl', 'rb') as file:
    X = pickle.load(file)

# Load the INTIME DataFrame from the pickle file
with open('data/INTIME.pkl', 'rb') as file:
    INTIME = pickle.load(file)

# Load dataset_detail
with open('data/dataset_detail.pkl', 'rb') as file:
    dataset_detail = pickle.load(file)

# Load target
with open('data/target.pkl', 'rb') as file:
    target = pickle.load(file)

In [226]:
dataset_detail.head()

,subject_id,hadm_id,stay_id,gender,admission_age,race,icu_intime,icu_outtime,anchor_age,anchor_year,...,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,edregtime,edouttime,intime
0,10000032,29079034,39553978,F,52.559969,WHITE,2180-07-23 14:00:00,2180-07-23 23:50:47,52,2180,...,EW EMER.,P30KEH,EMERGENCY ROOM,HOME,Medicaid,ENGLISH,WIDOWED,2180-07-23 05:54:00,2180-07-23 14:00:00,2180-07-23 14:00:00
1,10000980,26913865,39765666,F,76.486231,BLACK/AFRICAN AMERICAN,2189-06-27 08:42:00,2189-06-27 20:38:27,73,2186,...,EW EMER.,P30KEH,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,ENGLISH,MARRIED,2189-06-27 06:25:00,2189-06-27 08:42:00,2189-06-27 08:42:00
2,10001217,24597018,37067082,F,55.881486,WHITE,2157-11-20 19:18:02,2157-11-21 22:08:00,55,2157,...,EW EMER.,P4645A,EMERGENCY ROOM,HOME HEALTH CARE,Other,?,MARRIED,2157-11-18 17:38:00,2157-11-19 01:24:00,2157-11-20 19:18:02
3,10001217,27703517,34592300,F,55.962942,WHITE,2157-12-19 15:42:24,2157-12-20 14:27:41,55,2157,...,DIRECT EMER.,P99698,PHYSICIAN REFERRAL,HOME HEALTH CARE,Other,?,MARRIED,NaN,NaN,2157-12-19 15:42:24
4,10001725,25563031,31205490,F,46.275517,WHITE,2110-04-11 15:52:22,2110-04-12 23:59:56,46,2110,...,EW EMER.,P35SU0,PACU,HOME,Other,ENGLISH,MARRIED,NaN,NaN,2110-04-11 15:52:22


# Generate statistics

In [46]:
print("X shape", X.shape)
print("dataset_detail shape", dataset_detail.shape)

X shape (11163687, 31)
dataset_detail shape (73092, 22)


In [47]:
# id_list: list of unique icustay_id
dataset_detail_filtered = dataset_detail.loc[dataset_detail["stay_id"].isin(id_list)]
dataset_detail_filtered.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'gender', 'admission_age', 'race',
       'icu_intime', 'icu_outtime', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'deathtime', 'admission_type', 'admit_provider_id',
       'admission_location', 'discharge_location', 'insurance', 'language',
       'marital_status', 'edregtime', 'edouttime', 'intime'],
      dtype='object')

In [48]:
# X_copy_extract_label, dataset_detail
print("Merge X and dataset_detail")
merged_data = pd.merge(X, dataset_detail_filtered, on=["stay_id"], how="left", copy=False)
print("merged_data shape", merged_data.shape)

# Drop the columns subject_id_x and hadm_id_x
print("Drop duplicated columns")
merged_data.drop(columns=['subject_id_x', 'hadm_id_x'], inplace=True)

print("Rename the columns subject_id_y and hadm_id_y by removing the '_y'")
# Rename the columns subject_id_y and hadm_id_y by removing the '_y'
merged_data.rename(columns={'subject_id_y': 'subject_id', 'hadm_id_y': 'hadm_id'}, inplace=True)

# Display the updated columns
merged_data.columns

Merge X and dataset_detail
merged_data shape (11163687, 52)
Drop duplicated columns
Rename the columns subject_id_y and hadm_id_y by removing the '_y'


Index(['stay_id', 'charttime', 'creat_low_past_7day', 'creat_low_past_48hr',
       'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr', 'aki_stage_crrt', 'aki_stage',
       'aki_stage_smoothed', 'creat', 'aniongap_avg', 'bicarbonate_avg',
       'chloride_avg', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg',
       'sodium_avg', 'bun_avg', 'wbc_avg', 'glucose_avg', 'heartrate_mean',
       'sysbp_mean', 'diasbp_mean', 'resprate_mean', 'spo2_mean', 'vent',
       'vasopressor', 'sedative', 'subject_id', 'hadm_id', 'gender',
       'admission_age', 'race', 'icu_intime', 'icu_outtime', 'anchor_age',
       'anchor_year', 'anchor_year_group', 'deathtime', 'admission_type',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'intime'],
      dtype='object')

In [49]:
merged_data[['stay_id','charttime','icu_intime','icu_outtime','edregtime','edouttime','intime']].head()

,stay_id,charttime,icu_intime,icu_outtime,edregtime,edouttime,intime
0,30000153,2174-09-29 10:16:00,2174-09-29 12:09:00,2174-10-01 03:26:10,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00
1,30000153,2174-09-29 12:00:00,2174-09-29 12:09:00,2174-10-01 03:26:10,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00
2,30000153,2174-09-29 12:05:00,2174-09-29 12:09:00,2174-10-01 03:26:10,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00
3,30000153,2174-09-29 12:06:00,2174-09-29 12:09:00,2174-10-01 03:26:10,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00
4,30000153,2174-09-29 12:09:00,2174-09-29 12:09:00,2174-10-01 03:26:10,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00


In [50]:
merged_data.head()

,stay_id,charttime,creat_low_past_7day,creat_low_past_48hr,uo_rt_6hr,uo_rt_12hr,uo_rt_24hr,aki_stage_crrt,aki_stage,aki_stage_smoothed,...,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,edregtime,edouttime,intime
0,30000153,2174-09-29 10:16:00,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,EW EMER.,P462X1,EMERGENCY ROOM,REHAB,Other,ENGLISH,MARRIED,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00
1,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,EW EMER.,P462X1,EMERGENCY ROOM,REHAB,Other,ENGLISH,MARRIED,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00
2,30000153,2174-09-29 12:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,EW EMER.,P462X1,EMERGENCY ROOM,REHAB,Other,ENGLISH,MARRIED,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00
3,30000153,2174-09-29 12:06:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,EW EMER.,P462X1,EMERGENCY ROOM,REHAB,Other,ENGLISH,MARRIED,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00
4,30000153,2174-09-29 12:09:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,EW EMER.,P462X1,EMERGENCY ROOM,REHAB,Other,ENGLISH,MARRIED,2174-09-29 09:54:00,2174-09-29 12:09:00,2174-09-29 12:09:00


In [74]:
print("Shape of merged_data:", merged_data.shape)
print("Unique stay_id in merged_data:", merged_data['stay_id'].nunique())
print("Unique patiens in merged_data:", merged_data['subject_id'].nunique())

Shape of merged_data: (11163687, 50)
Unique stay_id in merged_data: 65413
Unique patiens in merged_data: 45233


In [52]:
# Group by subject_id and calculate the mean for each group.
# It is performed because a patient can have multiple stays.
patient_ages = merged_data[['anchor_age','subject_id']].groupby('subject_id').mean()
print(f"Age of patients: Avg: {patient_ages['anchor_age'].mean()} - Std: {patient_ages['anchor_age'].std()}")
print(f"Missing values in anchor_age: {merged_data['anchor_age'].isnull().sum()}")

Age of patients: Avg: 64.07140804280061 - Std: 16.706386916796422
Missing values in anchor_age: 0


In [54]:
# Group by subject_id and get the first occurrence of each subject to avoid counting duplicates
unique_patients_gender = dataset_detail_filtered.groupby('subject_id')['gender'].first()
unique_patients_race = dataset_detail_filtered.groupby('subject_id')['race'].first()

# Count the number of unique patients for each gender
gender_counts = unique_patients_gender.value_counts()
print(gender_counts)
print(f"Missing values in gender: {merged_data['gender'].isnull().sum()}")

gender
M    25449
F    19784
Name: count, dtype: int64
Missing values in gender: 0


In [55]:
# count the number of unique patients for each race
race_counts = unique_patients_race.value_counts()
print(race_counts)
print(f"Missing values in gender: {merged_data['race'].isnull().sum()}")

race
WHITE                                        29285
UNKNOWN                                       4652
BLACK/AFRICAN AMERICAN                        3431
OTHER                                         1547
WHITE - OTHER EUROPEAN                         833
UNABLE TO OBTAIN                               620
ASIAN                                          532
ASIAN - CHINESE                                495
HISPANIC/LATINO - PUERTO RICAN                 481
HISPANIC OR LATINO                             429
WHITE - RUSSIAN                                380
PATIENT DECLINED TO ANSWER                     326
HISPANIC/LATINO - DOMINICAN                    309
BLACK/CAPE VERDEAN                             275
BLACK/CARIBBEAN ISLAND                         263
BLACK/AFRICAN                                  171
ASIAN - SOUTH EAST ASIAN                       151
PORTUGUESE                                     149
ASIAN - ASIAN INDIAN                           112
WHITE - EASTERN EUROPEAN  

In [57]:
# Group by subject_id and get the first occurrence of each subject to avoid counting duplicates
icu_intime_ = merged_data.groupby('stay_id')['icu_intime'].first()
icu_outtime_ = merged_data.groupby('stay_id')['icu_outtime'].first()

print("Null values in icu_intime:", merged_data['icu_intime'].isnull().sum())
print("Null values in icu_outtime:", merged_data['icu_outtime'].isnull().sum())

Null values in icu_intime: 0
Null values in icu_outtime: 0


In [63]:
icu_intime_ = pd.to_datetime(icu_intime_)
icu_outtime_ = pd.to_datetime(icu_outtime_)
icu_duration = icu_outtime_ - icu_intime_

In [68]:
# Calculate the length of stay in days
length_of_stay = icu_duration.dt.total_seconds() / (3600 * 24)
length_of_stay

stay_id
30000153    1.636921
30000213    1.635278
30000484    2.478889
30000646    4.697523
30001148    1.135127
              ...   
39999286    1.157500
39999384    1.275764
39999552    1.167616
39999562    4.996296
39999810    4.743715
Length: 65413, dtype: float64

In [69]:
print(f"length_of_stay average {length_of_stay.mean()}")
print(f"length of stay std {length_of_stay.std()}")

length_of_stay average 3.7462867164953617
length of stay std 5.107577181678239


In [70]:
# Numbers generated from all the records
print(f"Nulls in admission_type:", merged_data['admission_type'].isnull().sum())
merged_data.groupby('stay_id')['admission_type'].first().value_counts()

Nulls in admission_type: 0


admission_type
EW EMER.                       34164
URGENT                         11518
OBSERVATION ADMIT               8301
SURGICAL SAME DAY ADMISSION     6531
DIRECT EMER.                    2506
ELECTIVE                        2275
EU OBSERVATION                    72
DIRECT OBSERVATION                42
AMBULATORY OBSERVATION             4
Name: count, dtype: int64

In [71]:
#Medications
print(f"Nulls in vasopressor:", merged_data['vasopressor'].isnull().sum())
print(f"vasopressor sum {merged_data['vasopressor'].sum()}\n")

print(f"Nulls in vent:", merged_data['vent'].isnull().sum())
print(f"Vent sum {merged_data['vent'].sum()}\n")

print(f"Nulls in sedative:", merged_data['sedative'].isnull().sum())
print(f"sedative sum {merged_data['sedative'].sum()}\n")

Nulls in vasopressor: 5731355
vasopressor sum 924650.0

Nulls in vent: 5731355
Vent sum 3055167.0

Nulls in sedative: 5731355
sedative sum 1337097.0



In [72]:
print("Nulls in creat", merged_data['creat'].isnull().sum())
print(f"Creatinine: Avg: {merged_data['creat'].mean()} - Std: {merged_data['creat'].std()}\n")

print("Nulls in uo_rt_6hr", merged_data['uo_rt_6hr'].isnull().sum())
print(f"Urine output at 6 hours: Avg: {merged_data['uo_rt_6hr'].mean()} - Std: {merged_data['uo_rt_6hr'].std()}\n")

print("Nulls in uo_rt_12hr", merged_data['uo_rt_12hr'].isnull().sum())
print(f"Urine output at 12 hours: Avg: {merged_data['uo_rt_12hr'].mean()} - Std: {merged_data['uo_rt_12hr'].std()}\n")

print("Nulls in uo_rt_24hr", merged_data['uo_rt_24hr'].isnull().sum())
print(f"Urine output at 24 hours: Avg: {merged_data['uo_rt_24hr'].mean()} - Std: {merged_data['uo_rt_24hr'].std()}\n")

print("Nulls in sodium_avg", merged_data['sodium_avg'].isnull().sum())
print(f"Sodium gap: Avg: {merged_data['sodium_avg'].mean()} - Std: {merged_data['sodium_avg'].std()}\n")

print("Nulls in potassium_avg", merged_data['potassium_avg'].isnull().sum())
print(f"Potassium gap: Avg: {merged_data['potassium_avg'].mean()} - Std: {merged_data['potassium_avg'].std()}\n")

print("Nulls in chloride_avg", merged_data['chloride_avg'].isnull().sum())
print(f"Chloride gap: Avg: {merged_data['chloride_avg'].mean()} - Std: {merged_data['chloride_avg'].std()}\n")

print("Nulls in aniongap_avg", merged_data['aniongap_avg'].isnull().sum())
print(f"Anion gap: Avg: {merged_data['aniongap_avg'].mean()} - Std: {merged_data['aniongap_avg'].std()}\n")

print("Nulls in bicarbonate_avg", merged_data['bicarbonate_avg'].isnull().sum())
print(f"Bicarbonate: Avg: {merged_data['bicarbonate_avg'].mean()} - Std: {merged_data['bicarbonate_avg'].std()}\n")

print("Nulls in bun_avg", merged_data['bun_avg'].isnull().sum())
print(f"Blood Urena nitrogen: Avg: {merged_data['bun_avg'].mean()} - Std: {merged_data['bun_avg'].std()}\n")

print("Nulls in glucose_avg", merged_data['glucose_avg'].isnull().sum())
print(f"Glucose: Avg: {merged_data['glucose_avg'].mean()} - Std: {merged_data['glucose_avg'].std()}\n")

print("Nulls in hematocrit_avg", merged_data['hematocrit_avg'].isnull().sum())
print(f"Hematocrit: Avg: {merged_data['hematocrit_avg'].mean()} - Std: {merged_data['hematocrit_avg'].std()}\n")

print("Nulls in hemoglobin_avg", merged_data['hemoglobin_avg'].isnull().sum())
print(f"Hemoglobin: Avg: {merged_data['hemoglobin_avg'].mean()} - Std: {merged_data['hemoglobin_avg'].std()}\n")

print("Nulls in wbc_avg", merged_data['wbc_avg'].isnull().sum())
print(f"White blood cells: Avg: {merged_data['wbc_avg'].mean()} - Std: {merged_data['wbc_avg'].std()}\n")

Nulls in creat 10076325
Creatinine: Avg: 1.541652589784571 - Std: 1.5467819834856338

Nulls in uo_rt_6hr 8236043
Urine output at 6 hours: Avg: 1.17592916549963 - Std: 2.403969955780811

Nulls in uo_rt_12hr 8406988
Urine output at 12 hours: Avg: 1.1472291462361317 - Std: 2.1831935414589467

Nulls in uo_rt_24hr 8814300
Urine output at 24 hours: Avg: 1.1475116699377312 - Std: 2.0764928933067224

Nulls in sodium_avg 10006256
Sodium gap: Avg: 138.2098866023115 - Std: 5.676010776251278

Nulls in potassium_avg 9899398
Potassium gap: Avg: 4.139334693254466 - Std: 0.6372802658058925

Nulls in chloride_avg 10043929
Chloride gap: Avg: 102.32028286469041 - Std: 6.789487415718323

Nulls in aniongap_avg 10143355
Anion gap: Avg: 14.133112555521144 - Std: 3.949365178346182

Nulls in bicarbonate_avg 10140356
Bicarbonate: Avg: 25.092709299337162 - Std: 5.096770366147789

Nulls in bun_avg 10135095
Blood Urena nitrogen: Avg: 30.92211586323829 - Std: 24.373193738396374

Nulls in glucose_avg 9195546
Glucose

In [73]:
print("Vital signs statistics")

print('diasbp_mean')
print(merged_data['diasbp_mean'].mean(), merged_data['diasbp_mean'].std(), merged_data['diasbp_mean'].isna().sum())
print()

print('sysbp_mean')
print(merged_data['sysbp_mean'].mean(), merged_data['sysbp_mean'].std(), merged_data['sysbp_mean'].isna().sum())
print()

print('heartrate_mean')
print(merged_data['heartrate_mean'].mean(), merged_data['heartrate_mean'].std(), merged_data['heartrate_mean'].isna().sum())
print()

print('resprate_mean')
print(merged_data['resprate_mean'].mean(), merged_data['resprate_mean'].std(), merged_data['resprate_mean'].isna().sum())
print()

print('spo2_mean')
print(merged_data['spo2_mean'].mean(), merged_data['spo2_mean'].std(), merged_data['spo2_mean'].isna().sum())
print()

Vital signs statistics
diasbp_mean
62.77914666651626 15.215830065919057 5135399

sysbp_mean
119.64257572913658 22.618245078570382 5134021

heartrate_mean
86.34865423473872 18.30863189173154 4890431

resprate_mean
20.148651057222434 5.868035026105263 4890693

spo2_mean
96.77838046638628 3.266665324339711 5013120



# Continue here

In [75]:
print("dataset drop AKI stages column")
X = X.drop(["aki_stage"], axis=1)

dataset drop AKI stages column


In [76]:
X = X.drop(["subject_id", "hadm_id"], axis=1)

In [77]:
print(X.shape)
X.columns

(11163687, 28)


Index(['stay_id', 'charttime', 'creat_low_past_7day', 'creat_low_past_48hr',
       'uo_rt_6hr', 'uo_rt_12hr', 'uo_rt_24hr', 'aki_stage_crrt',
       'aki_stage_smoothed', 'creat', 'aniongap_avg', 'bicarbonate_avg',
       'chloride_avg', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg',
       'sodium_avg', 'bun_avg', 'wbc_avg', 'glucose_avg', 'heartrate_mean',
       'sysbp_mean', 'diasbp_mean', 'resprate_mean', 'spo2_mean', 'vent',
       'vasopressor', 'sedative'],
      dtype='object')

# Resampling

In [78]:
# label = ['aki_stage']
skip = ["stay_id", "charttime"]
if MAX_FEATURE_SET:
    discrete_feat = ["sedative", "vasopressor", "vent"]
    skip.extend(discrete_feat)
    # all features that are not in skip are numeric
    
numeric_feat = list(X.columns.difference(skip))

In [79]:
print("Discrete features: ", discrete_feat)
print("Numeric features: ", numeric_feat)

Discrete features:  ['sedative', 'vasopressor', 'vent']
Numeric features:  ['aki_stage_crrt', 'aki_stage_smoothed', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg', 'creat', 'creat_low_past_48hr', 'creat_low_past_7day', 'diasbp_mean', 'glucose_avg', 'heartrate_mean', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean', 'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr', 'uo_rt_6hr', 'wbc_avg']


In [80]:
print("TIME_SAMPLING", TIME_SAMPLING)
print("MOST_COMMON",MOST_COMMON)
print("SAMPLING_INTERVAL",SAMPLING_INTERVAL)
print("MAX_FEATURE_SET", MAX_FEATURE_SET)

TIME_SAMPLING True
MOST_COMMON False
SAMPLING_INTERVAL 1H
MAX_FEATURE_SET True


In [81]:
if TIME_SAMPLING and MOST_COMMON:
    print("resampling: MOST_COMMON with interval of " + str(SAMPLING_INTERVAL))
    # Resample the data using assigned interval,mode() for most common
    X = X.set_index("charttime").groupby("stay_id").resample(SAMPLING_INTERVAL).mode().reset_index()

elif TIME_SAMPLING:
    print("resampling: MEAN & ZERO with interval of " + str(SAMPLING_INTERVAL))
    # Sampling with different strategies per kind of variable
    # label = ['aki_stage']
    skip = ["stay_id", "charttime"] #'aki_stage'
    
    if MAX_FEATURE_SET:
        discrete_feat = ["sedative", "vasopressor", "vent"]
        # discrete_feat = ['sedative', 'vasopressor', 'vent', 'hadm_id'] # Like in original notebok
        skip.extend(discrete_feat)
    # all features that are not in skip are numeric
    numeric_feat = list(X.columns.difference(skip))

    # Applying aggregation to features depending on their type
    X = X.set_index("charttime").groupby("stay_id").resample(SAMPLING_INTERVAL)
    if MAX_FEATURE_SET:
        X_discrete = X[discrete_feat].max().fillna(FILL_VALUE).astype(np.int64)
    X_numeric = X[numeric_feat].mean()
    # X_label = X['aki_stage'].max()
    print("Merging sampled features")

    try:
        X = pd.concat([X_numeric, X_discrete], axis=1).reset_index() #X_label
    except:
        print("Exception")
        # X = pd.concat([X_numeric,X_label], axis=1).reset_index()
        X = X_numeric.reset_index()

print(X.shape)

# Label forward fill
# X['aki_stage'] = X['aki_stage'].ffill(limit=RESAMPLE_LIMIT)

resampling: MEAN & ZERO with interval of 1H
Merging sampled features
(18879678, 28)


# fit time span with Yereva

In [82]:
INTIME.head()

,stay_id,intime
0,39553978,2180-07-23 14:00:00
1,39765666,2189-06-27 08:42:00
2,37067082,2157-11-20 19:18:02
3,34592300,2157-12-19 15:42:24
4,31205490,2110-04-11 15:52:22


In [83]:
print("Merging intime column to X")
print("Drop rows that has HOURS > 48h, could be other time span. And drop rows that has HOURS < 0")
X = pd.merge(X, INTIME, on=["stay_id"], how="left", copy=False)
X["HOURS"] = (X.charttime - X.intime).apply(lambda s: s / np.timedelta64(1, "s")) / 60.0 / 60
X = X[X["HOURS"] >= 0]
X = X[X["HOURS"] <= MAX_HOUR]
X = X.reset_index(drop=True)

Merging intime column to X
Drop rows that has HOURS > 48h, could be other time span. And drop rows that has HOURS < 0


In [84]:
X.shape

(3079482, 30)

# Imputing 

In [85]:
X.columns

Index(['stay_id', 'charttime', 'aki_stage_crrt', 'aki_stage_smoothed',
       'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg', 'creat',
       'creat_low_past_48hr', 'creat_low_past_7day', 'diasbp_mean',
       'glucose_avg', 'heartrate_mean', 'hematocrit_avg', 'hemoglobin_avg',
       'potassium_avg', 'resprate_mean', 'sodium_avg', 'spo2_mean',
       'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr', 'uo_rt_6hr', 'wbc_avg',
       'sedative', 'vasopressor', 'vent', 'intime', 'HOURS'],
      dtype='object')

In [86]:
print("IMPUTE_EACH_ID", IMPUTE_EACH_ID)
print("IMPUTE_COLUMN", IMPUTE_COLUMN)

IMPUTE_EACH_ID True
IMPUTE_COLUMN False


In [87]:
print("Imputation.")
# X['aki_stage'] = X['aki_stage'].fillna(0)
remove_list = ["stay_id", "charttime", "intime", "HOURS"]

# using most common within each stay_id
if IMPUTE_EACH_ID:
    column_name = list(X.columns)
    for item in remove_list:
        column_name.remove(item)
    # column_name.remove(column_name[0]) 
    for feature in column_name:
        X.loc[X[feature].isnull(), feature] = X.stay_id.map(
            # Calculate the mode of the feature for each stay_id
            fast_mode(X, ["stay_id"], feature).set_index("stay_id")[feature]
        )

# imputation based on whole column
if IMPUTE_COLUMN:
    imp = SimpleImputer(missing_values=np.nan, strategy=IMPUTE_METHOD)
    cols = list(X.columns)
    for item in remove_list:
        cols.remove(item)
    X[cols] = imp.fit_transform(X[cols])

# If no imputation method selected or only impute each id, for the remaining nan impute direclty with FILL_VALUE
X = X.fillna(FILL_VALUE)

Imputation.


In [88]:
# more comfortable to review in this order
print("check variables")
try:
    cols = ['stay_id', 'charttime','aniongap_avg','bicarbonate_avg', 'bun_avg','chloride_avg',
            'creat','diasbp_mean', 'glucose_avg', 'heartrate_mean', 'hematocrit_avg','hemoglobin_avg', 
            'potassium_avg', 'resprate_mean', 'sodium_avg','spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 
            'uo_rt_24hr', 'uo_rt_6hr','wbc_avg', 'sedative', 'vasopressor', 'vent',"HOURS" , "intime"]
    X = X[cols]
    print("success")
except:
    try:
        cols = ['stay_id', 'charttime','aki_stage','creat','uo_rt_12hr', 'uo_rt_24hr', 'uo_rt_6hr']
        X = X[cols]
        print("try")
    except:
        print("error")

check variables
success


In [89]:
X.head()

,stay_id,charttime,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,...,sysbp_mean,uo_rt_12hr,uo_rt_24hr,uo_rt_6hr,wbc_avg,sedative,vasopressor,vent,HOURS,intime
0,30000153,2174-09-29 13:00:00,12.0,19.0,22.0,110.0,0.8,74.5,158.0,104.0,...,132.0,0.5321,0.5827,0.7045,11.9,1,0,1,0.85,2174-09-29 12:09:00
1,30000153,2174-09-29 14:00:00,12.0,19.0,22.0,112.0,0.8,61.0,176.0,83.0,...,131.0,0.5321,0.5827,0.7045,11.9,0,0,1,1.85,2174-09-29 12:09:00
2,30000153,2174-09-29 15:00:00,12.0,19.0,22.0,115.0,0.9,65.0,192.0,92.0,...,123.0,0.5321,0.5827,0.7045,17.0,1,0,1,2.85,2174-09-29 12:09:00
3,30000153,2174-09-29 16:00:00,12.0,19.0,22.0,115.0,0.8,55.0,175.0,83.0,...,109.0,0.5321,0.5827,0.7045,11.9,1,0,1,3.85,2174-09-29 12:09:00
4,30000153,2174-09-29 17:00:00,12.0,19.0,22.0,115.0,0.8,56.0,98.0,103.0,...,111.0,0.5321,0.5827,0.7045,11.9,0,0,1,4.85,2174-09-29 12:09:00


# Shifting labels

In [90]:
#print("Shifting the labels 48 h") # by 8 position : 6h sampling*8=48h and ffil 8 newly empty ones
# X['aki_stage'] = X.groupby('stay_id')['aki_stage'].shift(-(HOURS_AHEAD // int(SAMPLING_INTERVAL[:-1])))
# X = X.dropna(subset=['aki_stage'])
# X['stay_id'].nunique()

# Add categorical features (details)

In [91]:
dataset_detail.head()

,subject_id,hadm_id,stay_id,gender,admission_age,race,icu_intime,icu_outtime,anchor_age,anchor_year,...,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,edregtime,edouttime,intime
0,10000032,29079034,39553978,F,52.559969,WHITE,2180-07-23 14:00:00,2180-07-23 23:50:47,52,2180,...,EW EMER.,P30KEH,EMERGENCY ROOM,HOME,Medicaid,ENGLISH,WIDOWED,2180-07-23 05:54:00,2180-07-23 14:00:00,2180-07-23 14:00:00
1,10000980,26913865,39765666,F,76.486231,BLACK/AFRICAN AMERICAN,2189-06-27 08:42:00,2189-06-27 20:38:27,73,2186,...,EW EMER.,P30KEH,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,ENGLISH,MARRIED,2189-06-27 06:25:00,2189-06-27 08:42:00,2189-06-27 08:42:00
2,10001217,24597018,37067082,F,55.881486,WHITE,2157-11-20 19:18:02,2157-11-21 22:08:00,55,2157,...,EW EMER.,P4645A,EMERGENCY ROOM,HOME HEALTH CARE,Other,?,MARRIED,2157-11-18 17:38:00,2157-11-19 01:24:00,2157-11-20 19:18:02
3,10001217,27703517,34592300,F,55.962942,WHITE,2157-12-19 15:42:24,2157-12-20 14:27:41,55,2157,...,DIRECT EMER.,P99698,PHYSICIAN REFERRAL,HOME HEALTH CARE,Other,?,MARRIED,NaN,NaN,2157-12-19 15:42:24
4,10001725,25563031,31205490,F,46.275517,WHITE,2110-04-11 15:52:22,2110-04-12 23:59:56,46,2110,...,EW EMER.,P35SU0,PACU,HOME,Other,ENGLISH,MARRIED,NaN,NaN,2110-04-11 15:52:22


In [92]:
dataset_detail.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'gender', 'admission_age', 'race',
       'icu_intime', 'icu_outtime', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'deathtime', 'admission_type', 'admit_provider_id',
       'admission_location', 'discharge_location', 'insurance', 'language',
       'marital_status', 'edregtime', 'edouttime', 'intime'],
      dtype='object')

In [93]:
# dataset_detail[['admission_age', 'anchor_age']].tail()
dataset_detail[['icu_intime','intime']].head()

,icu_intime,intime
0,2180-07-23 14:00:00,2180-07-23 14:00:00
1,2189-06-27 08:42:00,2189-06-27 08:42:00
2,2157-11-20 19:18:02,2157-11-20 19:18:02
3,2157-12-19 15:42:24,2157-12-19 15:42:24
4,2110-04-11 15:52:22,2110-04-11 15:52:22


In [94]:
dataset_detail_copy = dataset_detail.copy()

In [95]:
numeric_feat

['aki_stage_crrt',
 'aki_stage_smoothed',
 'aniongap_avg',
 'bicarbonate_avg',
 'bun_avg',
 'chloride_avg',
 'creat',
 'creat_low_past_48hr',
 'creat_low_past_7day',
 'diasbp_mean',
 'glucose_avg',
 'heartrate_mean',
 'hematocrit_avg',
 'hemoglobin_avg',
 'potassium_avg',
 'resprate_mean',
 'sodium_avg',
 'spo2_mean',
 'sysbp_mean',
 'uo_rt_12hr',
 'uo_rt_24hr',
 'uo_rt_6hr',
 'wbc_avg']

In [96]:
if MAX_FEATURE_SET:
    # extract datasets based on id_list
    dataset_detail = dataset_detail.loc[dataset_detail["stay_id"].isin(id_list)]
    # sort by ascending order
    dataset_detail = dataset_detail.sort_values(by=["stay_id"])
    # subject_id = dataset_detail["subject_id"].unique()
    # print(dataset_detail)

    # transfrom categorical data to binary form
    dataset_detail = dataset_detail.drop(["intime"], axis=1)
    dataset_detail = dataset_detail.join(pd.get_dummies(dataset_detail.pop("gender")))
    dataset_detail = dataset_detail.join(pd.get_dummies(dataset_detail.pop("race")))
    dataset_detail = dataset_detail.join(pd.get_dummies(dataset_detail.pop("admission_type")))
    # X = X.drop(['subject_id', 'hadm_id'], axis=1)
    # dataset_detail = dataset_detail.drop(['subject_id', 'hadm_id'], axis=1)
    X = pd.merge(X, dataset_detail, on=["stay_id"], how="left", copy=False)
    numeric_feat.append("anchor_age")

In [97]:
dataset_detail.columns.to_list()

['subject_id',
 'hadm_id',
 'stay_id',
 'admission_age',
 'icu_intime',
 'icu_outtime',
 'anchor_age',
 'anchor_year',
 'anchor_year_group',
 'deathtime',
 'admit_provider_id',
 'admission_location',
 'discharge_location',
 'insurance',
 'language',
 'marital_status',
 'edregtime',
 'edouttime',
 'F',
 'M',
 'AMERICAN INDIAN/ALASKA NATIVE',
 'ASIAN',
 'ASIAN - ASIAN INDIAN',
 'ASIAN - CHINESE',
 'ASIAN - KOREAN',
 'ASIAN - SOUTH EAST ASIAN',
 'BLACK/AFRICAN',
 'BLACK/AFRICAN AMERICAN',
 'BLACK/CAPE VERDEAN',
 'BLACK/CARIBBEAN ISLAND',
 'HISPANIC OR LATINO',
 'HISPANIC/LATINO - CENTRAL AMERICAN',
 'HISPANIC/LATINO - COLUMBIAN',
 'HISPANIC/LATINO - CUBAN',
 'HISPANIC/LATINO - DOMINICAN',
 'HISPANIC/LATINO - GUATEMALAN',
 'HISPANIC/LATINO - HONDURAN',
 'HISPANIC/LATINO - MEXICAN',
 'HISPANIC/LATINO - PUERTO RICAN',
 'HISPANIC/LATINO - SALVADORAN',
 'MULTIPLE RACE/ETHNICITY',
 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER',
 'OTHER',
 'PATIENT DECLINED TO ANSWER',
 'PORTUGUESE',
 'SOUTH AMERI

In [98]:
# If checkpoint 1 is loaded, the following cell should be executed to load the dataset_detail DataFrame.
# X = pd.merge(X, dataset_detail, on=["stay_id"], how="left", copy=False)
# numeric_feat.append("anchor_age")

In [99]:
X.columns.to_list()

['stay_id',
 'charttime',
 'aniongap_avg',
 'bicarbonate_avg',
 'bun_avg',
 'chloride_avg',
 'creat',
 'diasbp_mean',
 'glucose_avg',
 'heartrate_mean',
 'hematocrit_avg',
 'hemoglobin_avg',
 'potassium_avg',
 'resprate_mean',
 'sodium_avg',
 'spo2_mean',
 'sysbp_mean',
 'uo_rt_12hr',
 'uo_rt_24hr',
 'uo_rt_6hr',
 'wbc_avg',
 'sedative',
 'vasopressor',
 'vent',
 'HOURS',
 'intime',
 'subject_id',
 'hadm_id',
 'admission_age',
 'icu_intime',
 'icu_outtime',
 'anchor_age',
 'anchor_year',
 'anchor_year_group',
 'deathtime',
 'admit_provider_id',
 'admission_location',
 'discharge_location',
 'insurance',
 'language',
 'marital_status',
 'edregtime',
 'edouttime',
 'F',
 'M',
 'AMERICAN INDIAN/ALASKA NATIVE',
 'ASIAN',
 'ASIAN - ASIAN INDIAN',
 'ASIAN - CHINESE',
 'ASIAN - KOREAN',
 'ASIAN - SOUTH EAST ASIAN',
 'BLACK/AFRICAN',
 'BLACK/AFRICAN AMERICAN',
 'BLACK/CAPE VERDEAN',
 'BLACK/CARIBBEAN ISLAND',
 'HISPANIC OR LATINO',
 'HISPANIC/LATINO - CENTRAL AMERICAN',
 'HISPANIC/LATINO - COL

In [100]:
X.head()

,stay_id,charttime,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,...,WHITE - RUSSIAN,AMBULATORY OBSERVATION,DIRECT EMER.,DIRECT OBSERVATION,ELECTIVE,EU OBSERVATION,EW EMER.,OBSERVATION ADMIT,SURGICAL SAME DAY ADMISSION,URGENT
0,30000153,2174-09-29 13:00:00,12.0,19.0,22.0,110.0,0.8,74.5,158.0,104.0,...,False,False,False,False,False,False,True,False,False,False
1,30000153,2174-09-29 14:00:00,12.0,19.0,22.0,112.0,0.8,61.0,176.0,83.0,...,False,False,False,False,False,False,True,False,False,False
2,30000153,2174-09-29 15:00:00,12.0,19.0,22.0,115.0,0.9,65.0,192.0,92.0,...,False,False,False,False,False,False,True,False,False,False
3,30000153,2174-09-29 16:00:00,12.0,19.0,22.0,115.0,0.8,55.0,175.0,83.0,...,False,False,False,False,False,False,True,False,False,False
4,30000153,2174-09-29 17:00:00,12.0,19.0,22.0,115.0,0.8,56.0,98.0,103.0,...,False,False,False,False,False,False,True,False,False,False


In [101]:
X = X.drop(['charttime', 'intime'], axis=1)

In [102]:
feature_names = [
    "Anion gap",
    "Bicarbonate",
    "Blood Urea Nitrogen",
    "Chloride",
    "Creatinine",
    "Diastolic BP",
    "Glucose",
    "Heart rate",
    "Hematocrit",
    "Hemoglobin",
    "Potassium",
    "Respiratory rate",
    "Sodium",
    "Oxygen saturation",
    "Systolic BP",
    "Urine output 12h",
    "Urine output 24h",
    "Urine output 6h",
    "White cell count",
    "Sedative",
    "Vasopressor",
    "Ventilation",
    "Age",
    "Female gender",
    "Male gender",
    "Asian ethnicity",
    "Black ethnicity",
    "Hispanic ethnicity",
    "Native american",
    "Other ethnicity",
    "Ethnicity unknown",
    "White ethnicity",
    "Elective admission",
    "Emergency admission",
    "Urgent admission",
]
print(len(feature_names))

35


# Cap features between 0.01 / 0.99 quantile and normalisation

In [103]:
numeric_feat

['aki_stage_crrt',
 'aki_stage_smoothed',
 'aniongap_avg',
 'bicarbonate_avg',
 'bun_avg',
 'chloride_avg',
 'creat',
 'creat_low_past_48hr',
 'creat_low_past_7day',
 'diasbp_mean',
 'glucose_avg',
 'heartrate_mean',
 'hematocrit_avg',
 'hemoglobin_avg',
 'potassium_avg',
 'resprate_mean',
 'sodium_avg',
 'spo2_mean',
 'sysbp_mean',
 'uo_rt_12hr',
 'uo_rt_24hr',
 'uo_rt_6hr',
 'wbc_avg',
 'anchor_age']

In [104]:
X.columns

Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'HOURS',
       'subject_id', 'hadm_id', 'admission_age', 'icu_intime', 'icu_outtime',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'deathtime',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'F', 'M', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'HISPANIC OR LATINO',
       'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISP

In [105]:
X = cap_data(X)

Capping between the 0.01 and 0.99 quantile


In [106]:
features_to_remove = ['aki_stage_crrt', 'aki_stage_smoothed', 'creat_low_past_48hr', 'creat_low_past_7day']
numeric_feat = [feature for feature in numeric_feat if feature not in features_to_remove]
numeric_feat

['aniongap_avg',
 'bicarbonate_avg',
 'bun_avg',
 'chloride_avg',
 'creat',
 'diasbp_mean',
 'glucose_avg',
 'heartrate_mean',
 'hematocrit_avg',
 'hemoglobin_avg',
 'potassium_avg',
 'resprate_mean',
 'sodium_avg',
 'spo2_mean',
 'sysbp_mean',
 'uo_rt_12hr',
 'uo_rt_24hr',
 'uo_rt_6hr',
 'wbc_avg',
 'anchor_age']

In [107]:
X = normalise_data(X, numeric_feat)

In [108]:
# X.loc[X["stay_id"] == 272725].sort_values(by=["HOURS"])["HOURS"]
X.columns

Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'HOURS',
       'subject_id', 'hadm_id', 'admission_age', 'icu_intime', 'icu_outtime',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'deathtime',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'F', 'M', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'HISPANIC OR LATINO',
       'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISP

In [109]:
# X = X.sort_values(by=['stay_id', 'charttime'])
seq_lengths = X.groupby(["stay_id"], as_index=False).size().sort_values(by=["size"], ascending=False)
sequence_length = seq_lengths.max()  # the longest sequence per icustay-id
print(sequence_length)

stay_id    39899718
size          30836
dtype: int64


In [110]:
# AL re-write as try except to make it work as hadm_id is not used if only one csv file is used and none are merged
try:
    X.drop(["hadm_id"], axis=1, inplace=True)
except:
    pass

In [111]:
X = X.sort_values(by=["subject_id", "HOURS"])
# features = X.shape[1]-3
# features

# Count number of variables for final dataset

In [112]:
feature_names

['Anion gap',
 'Bicarbonate',
 'Blood Urea Nitrogen',
 'Chloride',
 'Creatinine',
 'Diastolic BP',
 'Glucose',
 'Heart rate',
 'Hematocrit',
 'Hemoglobin',
 'Potassium',
 'Respiratory rate',
 'Sodium',
 'Oxygen saturation',
 'Systolic BP',
 'Urine output 12h',
 'Urine output 24h',
 'Urine output 6h',
 'White cell count',
 'Sedative',
 'Vasopressor',
 'Ventilation',
 'Age',
 'Female gender',
 'Male gender',
 'Asian ethnicity',
 'Black ethnicity',
 'Hispanic ethnicity',
 'Native american',
 'Other ethnicity',
 'Ethnicity unknown',
 'White ethnicity',
 'Elective admission',
 'Emergency admission',
 'Urgent admission']

In [113]:
print(len(X.columns))
X.columns

84


Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'HOURS',
       'subject_id', 'admission_age', 'icu_intime', 'icu_outtime',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'deathtime',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'F', 'M', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'HISPANIC OR LATINO',
       'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISPANIC/LATINO

In [114]:
white_group = ['WHITE', 'WHITE - OTHER EUROPEAN', 'WHITE - RUSSIAN', 'WHITE - EASTERN EUROPEAN', 'WHITE - BRAZILIAN']
black_african_group = ['BLACK/AFRICAN AMERICAN', 'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'BLACK/AFRICAN']
asian_group = ['ASIAN', 'ASIAN - CHINESE', 'ASIAN - SOUTH EAST ASIAN', 'ASIAN - ASIAN INDIAN', 'ASIAN - KOREAN']
hispanic_group = ['HISPANIC OR LATINO', 'HISPANIC/LATINO - PUERTO RICAN', 'HISPANIC/LATINO - DOMINICAN', 'HISPANIC/LATINO - GUATEMALAN', 'HISPANIC/LATINO - SALVADORAN', 'HISPANIC/LATINO - MEXICAN', 'HISPANIC/LATINO - CUBAN', 'HISPANIC/LATINO - COLUMBIAN', 'HISPANIC/LATINO - HONDURAN', 'HISPANIC/LATINO - CENTRAL AMERICAN']
unknown_group = ['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER']
native_group = ['AMERICAN INDIAN/ALASKA NATIVE', 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER']
other_group = ['OTHER', 'PORTUGUESE', 'SOUTH AMERICAN','MULTIPLE RACE/ETHNICITY']

# Create binary columns for each group
X['white_group'] = X[white_group].any(axis=1).astype(int)
X['black_african_group'] = X[black_african_group].any(axis=1).astype(int)
X['asian_group'] = X[asian_group].any(axis=1).astype(int)
X['hispanic_group'] = X[hispanic_group].any(axis=1).astype(int)
X['unknown_group'] = X[unknown_group].any(axis=1).astype(int)
X['native_group'] = X[native_group].any(axis=1).astype(int)
X['other_group'] = X[other_group].any(axis=1).astype(int)

In [115]:
X.columns

Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'HOURS',
       'subject_id', 'admission_age', 'icu_intime', 'icu_outtime',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'deathtime',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'F', 'M', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'HISPANIC OR LATINO',
       'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISPANIC/LATINO

In [116]:
features_to_extract = ['stay_id', 'subject_id','aniongap_avg', 'bicarbonate_avg', #,'charttime','aki_stage'
       'bun_avg', 'chloride_avg', 'creat', 'diasbp_mean', 'glucose_avg',
       'heartrate_mean', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg',
       'resprate_mean', 'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr',
       'uo_rt_24hr', 'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent',
       'anchor_age','F', 'M', 'ASIAN', 'BLACK/AFRICAN', 'HISPANIC OR LATINO', 'AMERICAN INDIAN/ALASKA NATIVE',
       'OTHER', 'UNKNOWN', 'WHITE', 'ELECTIVE', 'URGENT','HOURS']

X_fts_extract = X[features_to_extract]

In [117]:
features_to_extract_race_groups = ['stay_id','subject_id','aniongap_avg', 'bicarbonate_avg', #'charttime','aki_stage', 
       'bun_avg', 'chloride_avg', 'creat', 'diasbp_mean', 'glucose_avg',
       'heartrate_mean', 'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg',
       'resprate_mean', 'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr',
       'uo_rt_24hr', 'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent',
       'anchor_age','F', 'M', 'white_group','black_african_group', 'asian_group', 'hispanic_group', 'unknown_group',
       'native_group','other_group', 'ELECTIVE', 'URGENT','HOURS']

X_fts_extract_race_groups = X[features_to_extract_race_groups]

In [118]:
features_list = list(X.columns)

# list of variables to be removed at the end
remove_list_final = ["stay_id", "subject_id"] # exluded: "F"
for item in remove_list_final:
    features_list.remove(item)

features = len(features_list)
print("number of features: " + str(features))

number of features: 89


# Split dataset

In [97]:
# len(id_list)
# id_list

In [98]:
# print("divide dataset into train, test and validation sets")
# id_train, id_test_val = train_test_split(id_list, test_size = SPLIT_SIZE, random_state = RANDOM_SEED) # train set is 80%)
# print("train is %d" % len(id_train))

# # remaining 20% split in halves as test and validation 10% and 10%
# id_valid, id_test = train_test_split(id_test_val, test_size = 0.5, random_state = RANDOM_SEED) # test 10% valid 10%
# print("val and test are %d" %len(id_test))

In [99]:
# train = X[X.stay_id.isin(id_train)].sort_values(by=['stay_id'])
# test = X[X.stay_id.isin(id_test)].sort_values(by=['stay_id'], ignore_index = True) 
# validation = X[X.stay_id.isin(id_valid)].sort_values(by=['stay_id']) 

# test = test.sort_values(by=['stay_id', 'charttime'], ignore_index = True)
# train = train.sort_values(by=['stay_id', 'charttime'], ignore_index = True)
# validation = validation.sort_values(by=['stay_id', 'charttime'], ignore_index = True)

# Random split subject ID into train(val), and test

In [119]:
print(RANDOM_SPLIT)
if RANDOM_SPLIT:
    print("extract subject_id list")
    subject_id = X["subject_id"].unique()
    subject_id = np.sort(subject_id)

    print("number of unique subject id: " + str(len(subject_id)))

True
extract subject_id list
number of unique subject id: 44335


In [120]:
if RANDOM_SPLIT:
    print("RANDOM SPLIT")
    print("divide dataset into train, test and validation sets")
    id_train_val, id_test = train_test_split(subject_id, test_size=0.1, random_state=RANDOM_SEED)  # train set is 80%)
    print("test is %d" % len(id_test))
    # remaining 20% split in halves as test and validation 10% and 10%
    id_train, id_val = train_test_split(id_train_val, test_size=0.111, random_state=RANDOM_SEED)  # test 10% valid 10%
    print("train is %d" % len(id_train))
    print("val is %d" % len(id_val))

    # sort list
    id_test.sort()
    id_train.sort()
    id_val.sort()

RANDOM SPLIT
divide dataset into train, test and validation sets
test is 4434
train is 35471
val is 4430


In [121]:
# Get stay_ids for id_test
print("ids for test set")

unique_stay_ids_test = X[X['subject_id'].isin(id_test)]['stay_id'].unique()
print(len(unique_stay_ids_test))

# Get the y_true values for the stay_ids in id_test
target[target['stay_id'].isin(unique_stay_ids_test)]['y_true'].value_counts()

ids for test set
6229


y_true
1    4245
0    1984
Name: count, dtype: int64

In [122]:
# Get stay_ids for id_val
print("ids for val set")

unique_stay_ids_val = X[X['subject_id'].isin(id_val)]['stay_id'].unique()
print(len(unique_stay_ids_val))

# Get the y_true values for the stay_ids in id_val
target[target['stay_id'].isin(unique_stay_ids_val)]['y_true'].value_counts()

ids for val set
6906


y_true
1    4719
0    2187
Name: count, dtype: int64

In [123]:
# Get stay_ids for id_train
print("ids for train set")

unique_stay_ids_train = X[X['subject_id'].isin(id_train)]['stay_id'].unique()
print(len(unique_stay_ids_train))

# Get the y_true values for the stay_ids in id_train
target[target['stay_id'].isin(unique_stay_ids_train)]['y_true'].value_counts()

ids for train set
50974


y_true
1    34787
0    16187
Name: count, dtype: int64

# Use fixed id_list from Yereva

In [124]:
print(FIXED)
if FIXED:
    # the Yereva id files pathes are here
    DATA_PATH_yereva_test = "data/id_list_yereva/test_listfile.csv"
    DATA_PATH_yereva_train = "data/id_list_yereva/train_listfile.csv"
    DATA_PATH_yereva_val = "data/id_list_yereva/val_listfile.csv"

    print("read csv files")
    # reading csv files
    yereva_test = pd.read_csv(DATA_PATH_yereva_test, sep=",")
    yereva_train = pd.read_csv(DATA_PATH_yereva_train, sep=",")
    yereva_val = pd.read_csv(DATA_PATH_yereva_val, sep=",")

    # convert to list
    yereva_test = yereva_test["notes"].tolist()
    yereva_train = yereva_train["notes"].tolist()
    yereva_val = yereva_val["notes"].tolist()

    yereva_test_subject = []
    yereva_train_subject = []
    yereva_val_subject = []

    for subject in yereva_test:
        yereva_test_subject.append(int(subject.split("_")[0]))
    for subject in yereva_train:
        yereva_train_subject.append(int(subject.split("_")[0]))
    for subject in yereva_val:
        yereva_val_subject.append(int(subject.split("_")[0]))

False


In [125]:
if FIXED:
    id_test = []
    id_train = []
    id_val = []

    n = 0

    while n < len(subject_id):
        if subject_id[n] in yereva_test_subject:
            id_test.append(subject_id[n])
            n = n + 1
        elif subject_id[n] in yereva_train_subject:
            id_train.append(subject_id[n])
            n = n + 1
        elif subject_id[n] in yereva_val_subject:
            id_val.append(subject_id[n])
            n = n + 1
        else:
            n = n + 1

    id_test.sort()
    id_train.sort()
    id_val.sort()

    print("Fixed list from Yereva")
    print("test is %d" % len(id_test))
    print("train is %d" % len(id_train))
    print("val is %d" % len(id_val))

In [126]:
if FIXED:
    print("combine subject_id list")
    subject_id = id_test + id_train
    subject_id = subject_id + id_val
    subject_id.sort()

    print("number of unique subject id: " + str(len(subject_id)))

# Convert icustay data into individual timeseries csv

In [307]:
def convert_icustay_to_AKIfolder(dataset, subject_id, output_path, id_train, id_test, id_val, target):

    temp_icustay_list = []  # to store the stay_id under same subject_id
    n = 0  # index to loop through temp_icustay_liremove_list_finalst
    num_stay = 0
    # dataset = X
    temp_dataset = pd.DataFrame()
    sub_temp_dataset = pd.DataFrame()

    train_pairs = []
    test_pairs = []
    val_pairs = []

    for subject in subject_id:
        # make path for subject folder
        dn = os.path.join(output_path, str(subject))
        try:
            os.makedirs(dn)
        except:
            pass

        temp_dataset = dataset.loc[dataset["subject_id"] == subject].sort_values(by=["stay_id"])
        temp_icustay_list = temp_dataset["stay_id"].unique()
        num_stay = len(temp_icustay_list)
        print(f"Number of unique stay_id of patient {subject}: {num_stay}")
        n = 0

        while n < num_stay:
            sys.stdout.write(
                "\rSUBJECT_ID: {0} STAY_ID: {1} Episode {2}...".format(subject, temp_icustay_list[n], n + 1)
            )

            
            sub_temp_dataset = temp_dataset.loc[temp_dataset["stay_id"] == temp_icustay_list[n]]
            sub_temp_dataset = sub_temp_dataset.drop(remove_list_final, axis=1)
            sub_temp_dataset = sub_temp_dataset.set_index('HOURS').sort_index(axis=0)

            # Each csv corresponds to a single episode of a patient. An episode is defined as a single stay in the ICU.
            sub_temp_dataset.to_csv(
                os.path.join(
                    output_path,
                    str(subject),
                    "{}_episode{}_timeseries_{}.csv".format(subject, n + 1, temp_icustay_list[n]),
                ),
                index_label="Hours",
            )

            # create list for id list for train/test/val
            if subject in id_train:
                train_pairs.append(
                    (
                        str(subject) + "_note.txt",
                        str(subject) + "_episode" + str(n + 1) + "_timeseries_" + str(temp_icustay_list[n]) + ".csv",
                        str(list(target.loc[target["stay_id"] == temp_icustay_list[n]]["y_true"])[0]),
                    )
                )
            elif subject in id_test:
                test_pairs.append(
                    (
                        str(subject) + "_note.txt",
                        str(subject) + "_episode" + str(n + 1) + "_timeseries_" + str(temp_icustay_list[n]) + ".csv",
                        str(list(target.loc[target["stay_id"] == temp_icustay_list[n]]["y_true"])[0]),
                    )
                )
            elif subject in id_val:
                val_pairs.append(
                    (
                        str(subject) + "_note.txt",
                        str(subject) + "_episode" + str(n + 1) + "_timeseries_" + str(temp_icustay_list[n]) + ".csv",
                        str(list(target.loc[target["stay_id"] == temp_icustay_list[n]]["y_true"])[0]),
                    )
                )

            n = n + 1

    sys.stdout.write("DONE!\n")

    return train_pairs, test_pairs, val_pairs

In [308]:
remove_list_final

['stay_id', 'subject_id']

In [309]:
X_fts_extract.columns

Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'anchor_age',
       'F', 'M', 'ASIAN', 'BLACK/AFRICAN', 'HISPANIC OR LATINO',
       'AMERICAN INDIAN/ALASKA NATIVE', 'OTHER', 'UNKNOWN', 'WHITE',
       'ELECTIVE', 'URGENT'],
      dtype='object')

In [310]:
print(OUTPUT_PATH)
target

data/AKI_fts2


,stay_id,y_true
0,30000153,1
1,30000213,1
2,30000484,1
3,30000646,0
4,30001148,1
...,...,...
65408,39999286,1
65409,39999384,0
65410,39999552,0
65411,39999562,0


In [315]:
X.columns

Index(['stay_id', 'aniongap_avg', 'bicarbonate_avg', 'bun_avg', 'chloride_avg',
       'creat', 'diasbp_mean', 'glucose_avg', 'heartrate_mean',
       'hematocrit_avg', 'hemoglobin_avg', 'potassium_avg', 'resprate_mean',
       'sodium_avg', 'spo2_mean', 'sysbp_mean', 'uo_rt_12hr', 'uo_rt_24hr',
       'uo_rt_6hr', 'wbc_avg', 'sedative', 'vasopressor', 'vent', 'HOURS',
       'subject_id', 'admission_age', 'icu_intime', 'icu_outtime',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'deathtime',
       'admit_provider_id', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime',
       'F', 'M', 'AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN',
       'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN',
       'ASIAN - SOUTH EAST ASIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN',
       'BLACK/CAPE VERDEAN', 'BLACK/CARIBBEAN ISLAND', 'HISPANIC OR LATINO',
       'HISPANIC/LATINO - CENTRAL AMERICAN', 'HISPANIC/LATINO

In [329]:
OUTPUT_PATH_fts_extract_race_groups = os.path.join('data/AKI', "fts_extract_race_groups")

train_pairs, test_pairs, val_pairs = convert_icustay_to_AKIfolder(
    X_fts_extract_race_groups, subject_id, OUTPUT_PATH_fts_extract_race_groups, id_train, id_test, id_val, target
)

Number of unique stay_id of patient 10111112: 650
SUBJECT_ID: 10111112 STAY_ID: 39899718 Episode 650...Number of unique stay_id of patient 10111636: 1
SUBJECT_ID: 10111636 STAY_ID: 31318658 Episode 1...Number of unique stay_id of patient 10112163: 3
SUBJECT_ID: 10112163 STAY_ID: 39668021 Episode 3...Number of unique stay_id of patient 10112484: 1
SUBJECT_ID: 10112484 STAY_ID: 38897817 Episode 1...Number of unique stay_id of patient 10112789: 1
SUBJECT_ID: 10112789 STAY_ID: 32483147 Episode 1...Number of unique stay_id of patient 10112984: 1
SUBJECT_ID: 10112984 STAY_ID: 31093528 Episode 1...Number of unique stay_id of patient 10113381: 1
SUBJECT_ID: 10113381 STAY_ID: 36312173 Episode 1...Number of unique stay_id of patient 10113512: 1
SUBJECT_ID: 10113512 STAY_ID: 30119329 Episode 1...Number of unique stay_id of patient 10113636: 1
SUBJECT_ID: 10113636 STAY_ID: 30643549 Episode 1...Number of unique stay_id of patient 10113751: 2
SUBJECT_ID: 10113751 STAY_ID: 35339370 Episode 2...Number

# Move subject_timeseries.csv file to train/test folder 

In [326]:
def move_to_partition(subjects_root_path, patients, partition):
    if not os.path.exists(os.path.join(subjects_root_path, partition)):
        print(f"Creating  partition", os.path.join(subjects_root_path, partition))
        os.mkdir(os.path.join(subjects_root_path, partition))
    for patient in patients:
        src = os.path.join(subjects_root_path, str(patient))
        dest = os.path.join(subjects_root_path, partition)
        for filename in os.listdir(src):
            shutil.move(os.path.join(src, str(filename)), dest)
        os.rmdir(src)

In [330]:
move_to_partition(OUTPUT_PATH_fts_extract_race_groups, id_train, "train")
move_to_partition(OUTPUT_PATH_fts_extract_race_groups, id_val, "train")
move_to_partition(OUTPUT_PATH_fts_extract_race_groups, id_test, "test")

Creating  partition data/AKI\fts_extract_race_groups\train
Creating  partition data/AKI\fts_extract_race_groups\test


# Create test_listfile.csv, train_listfile.csv, val_listfile.csv, and move to AKI folder

In [331]:
with open(os.path.join(OUTPUT_PATH_fts_extract_race_groups, "train_listfile.csv"), "w") as listfile:
    listfile.write("notes,stay,y_true\n")
    for (n, x, y) in train_pairs:
        listfile.write("{},{},{}\n".format(n, x, str(y)))
        
with open(os.path.join(OUTPUT_PATH_fts_extract_race_groups, "val_listfile.csv"), "w") as listfile:
    listfile.write("notes,stay,y_true\n")
    for (n, x, y) in val_pairs:
        listfile.write("{},{},{}\n".format(n, x, str(y)))

with open(os.path.join(OUTPUT_PATH_fts_extract_race_groups, "test_listfile.csv"), "w") as listfile:
    listfile.write("notes,stay,y_true\n")
    for (n, x, y) in test_pairs:
        listfile.write("{},{},{}\n".format(n, x, str(y)))